# Week 4: Problém lineárních nejmenších čtverců

**Assignment:** [![Run in JupyterHub](https://img.shields.io/badge/Run%20in-JupyterHub-F37626?logo=jupyter)](https://hpc.troja.mff.cuni.cz:8000/hub/user-redirect/git-pull?app=lab&repo=https%3A%2F%2Fgitlab.karlin.mff.cuni.cz%2Fznm%2Fcviceni-student&branch=main&subpath=notebooks%2Fassignment%2Fweek-04-least-squares.ipynb) [![Download .ipynb](https://img.shields.io/badge/Download-.ipynb-blue)](https://gitlab.karlin.mff.cuni.cz/znm/cviceni-student/-/raw/main/notebooks/assignment/week-04-least-squares.ipynb?inline=false) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/num-math/cviceni-student/blob/main/notebooks/assignment/week-04-least-squares.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(123456789123456789)
rng_x_noise = np.random.default_rng(987654321987654321)
rng_y_noise = np.random.default_rng(456132789456123798)
import pandas as pd
import requests

### Materiály

- [poznámky z přednášky](https://www.karlin.mff.cuni.cz/~blechta/znm/assets/Approximacni_problemy.pdf)
- další materiály
  - kniha [Duintjer Tebbens a kolektiv: Analýza metod pro maticové výpočty](https://cuni.primo.exlibrisgroup.com/discovery/fulldisplay?docid=alma990020832750106986&context=L&vid=420CKIS_INST:UKAZ&lang=cs&search_scope=MyInst_and_CI&adaptor=Local%20Search%20Engine&tab=Everything&query=any,contains,Anal%C3%BDza%20metod%20pro%20maticov%C3%A9%20v%C3%BDpo%C4%8Dty&offset=0) (Kapitola 6)
  - skripta [Barto, Tůma: Lineární algebra](https://www.mff.cuni.cz/data/web/obsah/department_math/ka/skripta_la7.pdf) (Sekce 8.6)
  - skripta [Foundations of Applied Mathematics](https://foundations-of-applied-mathematics.github.io/) (Volume 1, Kapitola 5)

- ***Primární úlohy*** : B1, C1/C2

- ***Úlohy k bonusové části zkoušky*** : B1, C1

- základní ***Learning outcomes*** (cviko, nikoliv přednáška)
  - Umím odvodit formulaci pomocí normálních rovnic problému lineárních nejmenších čtverců.
  - Umím popsat alespoň 3 různé postupy numerického řešení problému lineárních nejmenších čtverců.
  - Umím formulovat problémy pro hledání parametrů modelů na základě naměřených dat ve smyslu problému nejmenších čtverců.

## Intro

V 19. století chtěl Gauss a "většina fyzikálně-matematického světa" přijít na to, kam se ztratila jistá plnetka *Ceres*, pro níž měli tehdy k dispozici několik pozorování.
Protože nebeská tělesa se "pohybují po elipsách", Gauss stál před problémem "najít tu správnou elipsu" na základě pozorování a v rámci tohoto cvičení odvodil metodu ***nejmenších čtverců*** (tento název pochází od Legendra, Gauss tu metodu nazval pouze jako "*principum nostrum*", tj. "*náš princip*").

V moderní matematice se k této metodě dá dojít skrze tzv. [***maximum likelihood estimate parametrů $\mathbf{\theta}$***; MLE](https://en.wikipedia.org/wiki/maximum_likelihood_estimation) - o tom se dozvíte více v příštím semestru na úvodním kurzu pravděpodobnosti a statistiky. Více detailů a další cool příklady lze najít v (českém) článku [200 let metody nejmenších čtverců](https://eudml.org/doc/196517).

My se spokojíme s algebraickou formulací tohoto problému, tj. hledáme vektor parametrů $\mathbf{\theta}\in \mathbb{R}^n$, který řeší

\begin{equation*}
\arg\min\limits_{\mathbf{\theta} \in \mathbb{R}^n} \,
\left\| \mathbf{b} - A \cdot \mathbf{\theta} \right\|^2,
\end{equation*}

pro dané

\begin{equation*}
A = \begin{bmatrix}
a_{11} & \dots & a_{1n} \\
a_{21} & \dots & a_{2n} \\
\vdots & \ddots & \vdots \\
\vdots & \ddots & \vdots \\
a_{m1} & \dots & a_{mn} \\
\end{bmatrix} \in \mathbb{R}^{m\times n}, \quad
\mathbf{b} =  \begin{bmatrix}
b_{1} \\
b_{2} \\
\vdots \\
\vdots \\
b_{m} \\
\end{bmatrix}, \quad
m \gg n.
\end{equation*}

***Poznámka*** : Neznámý vektor parametrů se klasicky značí jako $\mathbf{\theta}$ - ustálené značení, které uvidíte příští semestr. Abychom viděli souvislosti s řešením soustav rovnic se čtvercovou maticí, budeme místo značení neznámého vektoru $\mathbf{\theta}$ používat $\mathbf{x}$ a tedy náš problém bude formulován pro dané $A\in \mathbb{R}^{m\times n},\, \mathbf{b} \in \mathbb{R}^{m}$ jako

<a name="eq1"></a>
\begin{equation*}
\tag{1}
\arg\min\limits_{\mathbf{x} \in \mathbb{R}^n} \,
\left\| \mathbf{b} - A\mathbf{x} \right\|_2^2,
\end{equation*}

## Část A: Lineární nejmenší čtverce jako soustavy rovnic

Začneme odvozením toho, že [(1)](#eq1) opravdu vede na soustavu algebraických rovnic.

### Úloha A1 - normální rovnice

Jak bychom řešili [(1)](#eq1) se znalostmi z prvního ročníku? "Hledali bychom kandidáty na minimum jako body, kde derivace je nulová." Tak zkusíme, co vyjde, když to uděláme.

<div style="border-left: 4px solid #0969da; background-color: #ddf4ff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**📝 Tužka a papír**


Definujme $g \, : \, \mathbb{R}^n \rightarrow \mathbb{R}$ předpisem
  \begin{equation*}
    g( \mathbf{x} ) \, := \, \frac{1}{2}\| \mathbf{b}-A \mathbf{x} \|^2.
  \end{equation*}

- Ukažte, že $\nabla g(\mathbf{x}) = A^\top(A\mathbf{x} - \mathbf{b})$

Nechť $\mathbf{x}_* \in \mathbb{R}^n$ je libovolné takové, že $A^T(A \mathbf{x}_*-\mathbf{b})=0$. Ukažte, že
  - platí $g(\mathbf{x}_* + \mathbf{y}) - g(\mathbf{x}_*) = \frac{1}{2} \|A\mathbf{y}\|^2 $ pro každé $\mathbf{y}\in\mathbb{R}^n$
  - $\mathbf{x}_*$ je globální minimum $g$
  - pokud $A$ má lineárně nezávislé sloupce, pak $\mathbf{x}_*$ je ***jediné*** globální minimum $g$

</blockquote>
</div>

### Úloha A2 - rozšířená soustava

Úloha A1 ukazuje, že [(1)](#eq1) je ekvivalentní tzv. [problému normálních rovnic](https://mathworld.wolfram.com/NormalEquation.html), tj.

<a name="eq2"></a>
\begin{equation*}
\tag{2}
A^TA \mathbf{x}= A^T \mathbf{b}.
\end{equation*}

Další ekvivalentní zápis problému [(1)](#eq1) odpovídá řešení tzv. ***rozšířené soustavy***

<a name="eq3"></a>
\begin{equation*}
\tag{3}
\underbrace{ \begin{bmatrix} I & A \\ A^T & 0 \end{bmatrix} }_{=: \,C}
\begin{bmatrix} \mathbf{r} \\ \mathbf{x} \end{bmatrix} = \begin{bmatrix} \mathbf{b} \\ 0 \end{bmatrix},
\end{equation*}

kde $C \in \mathbb{R}^{(m+n) \times (m+n)}$.

<div style="border-left: 4px solid #0969da; background-color: #ddf4ff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**📝 Tužka a papír**

- Ukažte, že matice $C$ je regulární právě tehdy, když $A$ má plnou sloupcovou hodnost.
- Ukažte, že problémy [(2)](#eq2) a [(3)](#eq3) jsou ekvivalentní, tj. podvektor $\mathbf{x}$ řešení [(3)](#eq3) se rovná [(2)](#eq2). Čemu odpovídá podvektor $\mathbf{r}$ řešení [(3)](#eq3)?

</blockquote>
</div>

<div style="border-left: 4px solid #8250df; background-color: #fbefff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**❗ ZÚ (semestrální projekt) - podmíněnost rozšířené soustavy**

Cílem rozšířené soustavy je vyhnout se hlavnímu problému soustavy normálních rovnic: výraznému zhoršení podmíněnosti oproti původnímu problému.
- Vypočítejte číslo podmíněnosti matice $C$ v závislosti na singulárních číslech matice $A$.
  - Nejprve si vyjádřete největší/nejmenší singulární čísla matice $C$ v závislosti na matici $A$.
- Uvažujme 2 scénáře, ve kterých je $A$ špatně podmíněná, tj. $\sigma_{max}/\sigma_{min}$ je velké číslo:
  - *(i)* $\sigma_{max} \gg 1$ a zároveň $\sigma_{min} \approx 1$
  - *(ii)* $\sigma_{max} \approx 1$ a zároveň $\sigma_{min} \ll 1$

  Pokud bychom měli na výběr pouze mezi postupy [(2)](#eq2) a [(3)](#eq3) pro řešení problému $A\mathbf{x}=\mathbf{b}$, rozhodněte kterou z metod použít pro každý z případů a vysvětlete proč.
- [Nápověda & hinty](#head-w04-hinty-zua)

</blockquote>
</div>

### Opáčko z minule

Uvažujme problém [(1)](#eq1), kde $A\in \mathbb{R}^{m\times n}$, $m\gg n$ a matice $A$ má plnou sloupcovou hodnost.

Na přednášce (a částečně na [předchozím cviku](week-03-qr-decomposition)) jsme viděli, že [(1)](#eq1) lze také ekvivalentně formulovat pomocí matic z její $QR$-faktorizace. Konkrétně, označme-li si její $QR$-faktorizaci jako $A=QR$, pak stačí řešit rovnici

\begin{equation*}R\mathbf{x} = Q^T \mathbf{b}.\end{equation*}

Ve většině případů je navíc efektivnější počítat $QR$-faktorizaci matice $[A|\mathbf{b}]$, tj. $ \tilde{Q} \tilde{R} =  [A|\mathbf{b}]$ a následně řešit soustavu

\begin{equation*} \tilde{R}_{1:n,1:n} \mathbf{x} = \tilde{R}_{1:n,n+1}. \end{equation*}

## Část B: Lineární nejmenší čtverce a výpočet

Stejně jako u [Cvika 3](week-03-qr-decomposition) jsme výše odvodili několik ekvivalentních formulací problému [(1)](#eq1) a otázka je, kterou z nich použít jako numerickou metodu pro praktické řešení [(1)](#eq1). Naši adepti na postup řešení jsou

- spočítat $QR$-faktorizaci $QR = A$ a poté vyřešit problém $R_{1:n,1:n} \mathbf{x} = \left( Q^T\mathbf{b} \right)_{1:n}$,
- spočítat QR faktorizaci $\tilde{Q}\tilde{R} = [A|\mathbf{b}]$ a poté vyřešit problém $\tilde{R}_{1:n,1:n} \mathbf{x} = \tilde{R}_{n+1,1:n}$,
- sestavit a vyřešit soustavu $A^TA \mathbf{x} = A^T\mathbf{b}$,
- sestavit a vyřešit soustavu $\begin{bmatrix} I & A \\ A^T & 0 \end{bmatrix} \begin{bmatrix} \mathbf{r} \\ \mathbf{x} \end{bmatrix} = \begin{bmatrix} \mathbf{b} \\ 0 \end{bmatrix}$,
- spočítat ekonomický SVD rozklad $A = U_r\Sigma_r V_r^T$ (kde $\Sigma_r \in \mathbb{R}^{r\times r}$ je regulární a $r$ je rank matice $A$) a pak sestavit řešení jako $ \mathbf{x} = V_r \Sigma_r^{-1} U_r^T \mathbf{b}$ (matici $V_r \Sigma_r^{-1} U_r^T$ se říká tzv. [pseudoinverze, aka Moore-Penrose inverze $A$](https://en.wikipedia.org/wiki/Moore%E2%80%93Penrose_inverse#)).

Stejně jako u [Cvika 3](week-03-qr-decomposition) máme v principu 2 kritéria - kvalitu řešení a výpočetní náročnost.

### Úloha B1 - Kvalita

Pomocí níže uvedeného kódu testujte kvalitu numerických metod výše (5 metod) pro matice s fixní velikostí zvyšujícím se číslem podmíněnosti $\kappa(A)$.
  - matice `A` dimenze `m`-krat-`n` s číslem podmíněnosti `cond` je sestavena za pomoci předdefinované funkce `random_matrix(m,n,cond)`,
  - obrázky níže jsou vygenerovány se settingem `m=50`, `n=20` a `cond_numbers_to_try = np.logspace(0,20,30)`,
  - nejprve je předepsáno přesné řešení $\mathbf{x}$ (např. vektor $[1,\dotsc ,1]^T$) a poté nalezena vhodná pravá strana $\mathbf{b} = A\mathbf{x}$,
   - pro sestavování matic a vektorů "po blocích" je využita funkce [`np.block()`](https://numpy.org/doc/stable/reference/generated/numpy.block.html),
  - pro řešení soustav lineárních algebraických rovnic se čtvercovou maticí je použita funkce [`np.linalg.solve()`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.solve.html).
  - pro výpočet $QR$-faktorizace používáme funkci [`np.linalg.qr()`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.qr.html). Ta umožňuje i output pouze matice $R$.

<div style="border-left: 4px solid #0969da; background-color: #ddf4ff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**📝 Tužka a papír**

- Na základě výsledků/grafů odhadněte závislost kvality řešení (relativní reziduum & relativní chyba) na $\kappa(A)$ jako explicitní funkci.
- Pečlivě porovnejte s teorií z přednášky.
- Jakou máte intuici o stabilitě našich algoritmů na základě těchto experimentů?

</blockquote>
</div>

In [ ]:
def random_matrix(m, n, cond):
    """ Vrati nesingularni ctvercovou matici A rozmeru "m"-krat-"n" s cislem podminenosti kappa(A) = "cond". """
    rng = np.random.default_rng(123456789123456789) ### tenhle radek zajistuje, ze ty nahodnz cisla jsou vygenerovanz pri kazdem behu stejne (lze smazat)
    A = rng.standard_normal(size=(m,n))
    U, _, Vt = np.linalg.svd(A)
    singvals = np.logspace(0, -np.log10(cond), np.min([m,n]))
    S = np.zeros_like(A)
    np.fill_diagonal(S, singvals)
    return U @ S @ Vt
######################################################################################

### inicializace
m = 50
n = 20
cond_numbers_to_try = np.logspace(0, 12, 25)
e_rel_qr_a = np.zeros(len(cond_numbers_to_try))
r_rel_qr_a = np.zeros(len(cond_numbers_to_try))
e_rel_qr_ab = np.zeros(len(cond_numbers_to_try))
r_rel_qr_ab = np.zeros(len(cond_numbers_to_try))
e_rel_normal_eq = np.zeros(len(cond_numbers_to_try))
r_rel_normal_eq = np.zeros(len(cond_numbers_to_try))
e_rel_extended = np.zeros(len(cond_numbers_to_try))
r_rel_extended = np.zeros(len(cond_numbers_to_try))
e_rel_svd = np.zeros(len(cond_numbers_to_try))
r_rel_svd = np.zeros(len(cond_numbers_to_try))

for ind in range(len(cond_numbers_to_try)):
  A = random_matrix(m, n, cond_numbers_to_try[ind])
  x_exact = np.ones(n)
  b = A @ x_exact

  # qr(A)
  Q, R = np.linalg.qr(A)
  y = Q.T @ b
  x = np.linalg.solve(R[:n, :], y[:n])
  e_rel_qr_a[ind] = np.linalg.norm(x - x_exact) / np.linalg.norm(x_exact)
  r_rel_qr_a[ind] = np.linalg.norm(A @ x - b) / np.linalg.norm(b)

  # qr([A|b])
  Ab = np.block([A, b[:, np.newaxis]])
  Q, R = np.linalg.qr(Ab)
  x = np.linalg.solve(R[:n, :n], R[:n, n])
  e_rel_qr_ab[ind] = np.linalg.norm(x - x_exact) / np.linalg.norm(x_exact)
  r_rel_qr_ab[ind] = np.linalg.norm(A @ x - b) / np.linalg.norm(b)

  # A^T A
  x = np.linalg.solve(A.T @ A, A.T @ b)
  e_rel_normal_eq[ind] = np.linalg.norm(x - x_exact) / np.linalg.norm(x_exact)
  r_rel_normal_eq[ind] = np.linalg.norm(A @ x - b) / np.linalg.norm(b)

  # extended eqns
  C = np.block([ [np.eye(m),A],[A.T, np.zeros((n,n))] ])
  d = np.block( [b,np.zeros(n)] )
  extended_sol = np.linalg.solve(C,d)
  e_rel_extended[ind] = np.linalg.norm(x_exact-extended_sol[m:])/np.linalg.norm(x_exact)
  r_rel_extended[ind] = np.linalg.norm(extended_sol[0:m])/np.linalg.norm(b)

  # SVD
  U, S, Vt = np.linalg.svd(A, full_matrices=False)
  x = Vt.T @ ((U.T @ b) / S)
  e_rel_svd[ind] = np.linalg.norm(x - x_exact) / np.linalg.norm(x_exact)
  r_rel_svd[ind] = np.linalg.norm(A @ x - b) / np.linalg.norm(b)


############################## Vykreslovani
plt.figure(figsize=(15,6))
plt.subplot(1,2,1); plt.title(r'fixovane $m =$ {} a $n =$ {}'.format(m,n))
plt.loglog(cond_numbers_to_try,e_rel_qr_a, marker='o', markeredgecolor='k', label='$e_{rel}$ - qr($A$)')
plt.loglog(cond_numbers_to_try,e_rel_qr_ab, marker='o', markeredgecolor='k', label='$e_{rel}$ - qr($[A|b]$)')
plt.loglog(cond_numbers_to_try,e_rel_normal_eq, marker='o', markeredgecolor='k', label='$e_{rel}$ - $A^T A$')
plt.loglog(cond_numbers_to_try,e_rel_extended, marker='o', markeredgecolor='k', label=r'$e_{rel}$ - $C$')
plt.loglog(cond_numbers_to_try,e_rel_svd, marker='o', markeredgecolor='k', label=r'$e_{rel}$ - SVD')
plt.xlabel(r'$\kappa(A)$'); plt.legend(); # plt.xticks(np.logspace(0,20,30)); plt.xtick_labels(np.logspace(0,20,30));

plt.subplot(1,2,2); plt.title(r'fixovane $m =$ {} a $n =$ {}'.format(m,n))
plt.loglog(cond_numbers_to_try,r_rel_qr_a, marker='o', markeredgecolor='k', label='$r_{rel}$ - qr($A$)')
plt.loglog(cond_numbers_to_try,r_rel_qr_ab, marker='o', markeredgecolor='k', label='$r_{rel}$ - qr($[A|b]$)')
plt.loglog(cond_numbers_to_try,r_rel_normal_eq, marker='o', markeredgecolor='k', label='$r_{rel}$ - $A^T A$')
plt.loglog(cond_numbers_to_try,r_rel_extended, marker='o', markeredgecolor='k', label=r'$r_{rel}$ - $C$')
plt.loglog(cond_numbers_to_try,r_rel_svd, marker='o', markeredgecolor='k', label=r'$r_{rel}$ - SVD')
plt.xlabel(r'$\kappa(A)$'); plt.legend();
############################## Vykreslovani

### ZÚ - Výpočetní náročnost

Pro měření času programu použijeme funkci [`%timeit -o -q`](https://docs.python.org/3/library/timeit.html) :

  ```
  # nasledujici kod zmeri cas potrebny na exekuci funkce "my_function()"
  time_result = %timeit -o -q my_function_output = my_function(my_function_input)
  print('Prumerny cas na exekuci "my_function()" je', time_result.average)
  print('Smerodatna odchylka exekuce "my_function()" je', time_result.stdev)

  # Funkce "%timeit" mnohokrat spusti zbytek radku po prikazu "%timeit" a udela jednoduchou statistiku casu.
  # Meren je cas celeho zbyleho radku, tj. vseho za prikazem "%timeit".
  # Obecne funkce "%timeit" vraci objekt (tj. nikoliv pouze cas).
  # V tomto objektu je property "average" - tj. "time_result.average" udava prumerny cas na exekuci.
  # prikaz "-o" zajistuje, ze funkce "%timeit" da vystup
  # prikaz "-q" zajistuje, ze funkce "%timeit" nebude nic vypisovat
  ```

Pomocí níže uvedeného kódu otestujte časovou náročnost numerických metod výše (5 metod) pro náhodné matice s "rostoucí velikostí".
  - náhodná matice `A` dimenze `m`-krat-`n` je generována příkazem `A =  rng.standard_normal(size=(m,n))`,

  - pokud jde o velikost $A$, kód pracuje ve dvou módech:
    - *Mód 1* : Zafixujeme $m$ a postupně zvyšujeme $n$;
    - *Mód 2* : Zafixujeme poměr $m:n$ a postupně zvyšujeme $m$.

  - obrázky níže jsou vygenerovány se settingem
    - *Mód 1* : `m=1000`,
    - *Mód 2* : `m=2*n`.

<div style="border-left: 4px solid #8250df; background-color: #fbefff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**❗ ZÚ (semestrální projekt)**

- Za pomoci přednášky odhadněte asymptotickou výpočetní náročnost našich 5 metod (počet algebraických operací):
  - nejprve si rozmyslete nebo najděte-a-ocitujte kolik operací potřebujeme na standardní $QR$-faktorizaci matice $m$-krát-$n$,
  - pak si rozmyslete nebo najděte-a-ocitujte kolik operací potřebujeme na řešení čtvercové soustavy algebraických rovnic matice $n$-krát-$n$,
- Porovnejte výsledky s numerickými experimenty a vše zdůvodněte.

- [Nápověda & hinty](#head-w04-hinty-zu)

</blockquote>
</div>

**Poznámka 1** : Tenhle kód může být citlivý na ostatní aplikace na pozadí - vyplatí se vypnout další karty v prohlížeči či nepotřebné programy, pokud chceme replikovat obrázek níže.

**Poznámka 2** : S parametry napsanými výše kód poběží cca 10 minut, takže je vhodné si nastavit výrazně nižší hodnoty když debuggujete, např.:
  - `m=100` & `n_to_try=np.array([10,50,100])`,
  - `n_size_increases = 3`.

In [ ]:
m = 1000
b = np.random.randn(m) # 2-3mins pro m = 1000
n_to_try = np.array([100,250,500,750,1000])

times_qr_a = np.zeros(len(n_to_try))
times_qr_ab = np.zeros(len(n_to_try))
times_normal_eq = np.zeros(len(n_to_try))
times_extended = np.zeros(len(n_to_try))
times_svd = np.zeros(len(n_to_try))

for ind in range(0,len(n_to_try)):
  n = n_to_try[ind]
  A = random_matrix(m,n,10)

  time_result = %timeit -o -q q,r = np.linalg.qr(A); x_approx = np.linalg.solve(r[0:n,0:n], (q.T @ b)[0:n])
  times_qr_a[ind] = time_result.average

  Ab = np.hstack( (A,b.reshape(-1,1)) )
  time_result = %timeit -o -q r = np.linalg.qr(Ab,'r'); x_approx = np.linalg.solve(r[0:n,0:n],r[0:n,-1])
  times_qr_ab[ind] = time_result.average

  time_result = %timeit -o -q AtA = A.T @ A; x_approx = np.linalg.solve(AtA, A.T @ b)
  times_normal_eq[ind] = time_result.average

  time_result = %timeit -o -q C = np.block([ [np.eye(m),A],[A.T, np.zeros((n,n))] ]); d = np.block( [b,np.zeros(n)] ); extended_sol = np.linalg.solve(C,d)
  times_extended[ind] = time_result.average

  time_result = %timeit -o -q u,s,vt = np.linalg.svd(A); x_approx = vt.T @ (1/s * (u[:,0:n].T @ b))
  times_svd[ind] = time_result.average


plt.figure(figsize=(14,6))

plt.subplot(1,2,1); plt.title(r'fixovane $m =$ {} a variabilni $n$'.format(m))
plt.loglog(n_to_try,times_qr_a, marker='o', markeredgecolor='k', label='qr($A$)')
plt.loglog(n_to_try,times_qr_ab, marker='o', markeredgecolor='k', label='qr($[A|b]$)')
plt.loglog(n_to_try,times_normal_eq, marker='o', markeredgecolor='k', label='$A^T A$')
plt.loglog(n_to_try,times_extended, marker='o', markeredgecolor='k', label=r'$C$')
plt.loglog(n_to_try,times_svd, marker='o', markeredgecolor='k', label=r'SVD')
plt.xlabel(r'$n$'); plt.ylabel(r'cas'); plt.legend();


##########################################################################################################################################
n_size_increases = 6 # 4-5mins pro "8 zvetseni"
n_to_try = (50*2.0**(np.arange(0,n_size_increases,1))).astype(np.int32)
m_to_n_ratio = 2

times_qr_a = np.zeros(len(n_to_try))
times_qr_ab = np.zeros(len(n_to_try))
times_normal_eq = np.zeros(len(n_to_try))
times_extended = np.zeros(len(n_to_try))
times_svd = np.zeros(len(n_to_try))


for ind in range(0,len(n_to_try)):
  n = n_to_try[ind]
  m = m_to_n_ratio*n
  A = rng.standard_normal(size=(m,n))
  b = rng.standard_normal(size=m)

  time_result = %timeit -o -q q,r = np.linalg.qr(A); x_approx = np.linalg.solve(r[0:n,0:n], (q.T @ b)[0:n])
  times_qr_a[ind] = time_result.average

  Ab = np.hstack( (A,b.reshape(-1,1)) )
  time_result = %timeit -o -q r = np.linalg.qr(Ab,'r'); x_approx = np.linalg.solve(r[0:n,0:n],r[0:n,-1])
  times_qr_ab[ind] = time_result.average

  time_result = %timeit -o -q AtA = A.T @ A; x_approx = np.linalg.solve(AtA, A.T @ b)
  times_normal_eq[ind] = time_result.average

  time_result = %timeit -o -q C = np.block([ [np.eye(m),A],[A.T, np.zeros((n,n))] ]); d = np.block( [b,np.zeros(n)] ); extended_sol = np.linalg.solve(C,d)
  times_extended[ind] = time_result.average

  time_result = %timeit -o -q u,s,vt = np.linalg.svd(A); x_approx = vt.T @ (1/s * (u[:,0:n].T @ b))
  times_svd[ind] = time_result.average



plt.subplot(1,2,2); plt.title(r'fixovany pomer $m:n =$ {}:1 a variabilni $m$'.format(m_to_n_ratio))
plt.loglog(m_to_n_ratio*n_to_try,times_qr_a, marker='o', markeredgecolor='k', label='qr($A$)')
plt.loglog(m_to_n_ratio*n_to_try,times_qr_ab, marker='o', markeredgecolor='k', label='qr($[A|b]$)')
plt.loglog(m_to_n_ratio*n_to_try,times_normal_eq, marker='o', markeredgecolor='k', label='$A^T A$')
plt.loglog(m_to_n_ratio*n_to_try,times_extended, marker='o', markeredgecolor='k', label=r'$C$')
plt.loglog(m_to_n_ratio*n_to_try,times_svd, marker='o', markeredgecolor='k', label=r'SVD')
plt.xlabel(r'$m$'); plt.ylabel(r'cas'); plt.legend(); plt.show()

## Část C: Lineární nejmenší čtverce v praxi

Vyzkoušíme si pár aplikací vedoucích na lineárních problém nejmenších čtverců.

### Úloha C1 - Na které elipse jsem?

Podobně jako Gauss budeme hledat elipsu na základě (nepřesně) naměřených bodů, které na dané elipse leží. Gauss měl práci s hledáním "té právé elipsy" ztíženou faktem, že tu elipsu musel hledat v $\mathbb{R}^{3}$ - my si uděláme život snazší. Body i elipsu budeme uvažovat rovnou v $\mathbb{R}^{2}$.

Jedním ze způsobů jak zadat elipsu v rovině je jako implicitní funkci, konkrétně jako body $(x,y)$ splňující

<a name="eq4"></a>
\begin{equation*}
\tag{4}
ax^2 + bx + cxy + dy + ey^2 = 1,
\end{equation*}

pro dané koeficienty $a,b,c,d,e\in \mathbb{R}$ (koeficient $e$ v tomto značení "nemá nic společného" s excentricitou elipsy). Pro daný set bodů $(x_i,y_i)_{i=1,\dotsc, m}$ můžeme hledat elipsu, která nejlépe odpovídá těmto bodům, tj. koeficienty $a_{fit}, b_{fit},c_{fit}, d_{fit}, e_{fit}$, pro které je [(4)](#eq4) "splněno nejlépe, co to jde" (opět doporučujeme přečíst si ten článek [200 let metody nejmenších čtverců](https://eudml.org/doc/196517), který ty uvozovky vysvětluje/motivuje detailně).

<div style="border-left: 4px solid #0969da; background-color: #ddf4ff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**📝 Tužka a papír & kódění**

- **Tužka a papír**: Zformulujte matematickou úlohu pro nalezení nejlepších $a_{fit}, b_{fit},c_{fit}, d_{fit}, e_{fit}$ ve smyslu nejmenších čtverců, tj. jako problém $A\mathbf{x} = \mathbf{b}$ pro konkrétní $A$ a $\mathbf{b}$ dané body $(x_i,y_i)_{i=1,\dotsc, m}$.
  - [Nápověda & hinty](#head-w04-hinty-c1)

- **Kódění**: Pro elipsu $E$ danou koeficienty `exact_coeffs` nám kód níže vygeneruje $x$-ové a $y$*-ové souřadnice `x_exact, y_exact` bodů ležících na $E$ a také $x$-ové a $y$*-ové souřadnice `x_observed, y_observed` bodů, "které jsme pozorovali a jsou tedy nepřesné - zatížené chybou měření".

    - Sestavte a vyřešte problém $A \mathbf{x} = \mathbf{b}$, který odpovídá nalezení nejlépe fitující elipsy na základě "pozorovaných bodů" `x_observed, y_observed`.
    - pro sestavování matice po blocích (nebo po sloupcích/řádcích) se hodí funkce [`np.block()`](https://numpy.org/doc/stable/reference/generated/numpy.block.html),

    - řešení uložte do vektoru `fit_coeffs = [a_fit, b_fit,c_fit,d_fit,e_fit]` pro hladký průběh následného vykreslování.

</blockquote>
</div>

In [ ]:
def generate_exact_ellipse_points(coeffs, n_points, plot=False):
  """Vygeneruje "n_points" bodu na elipse dane rovnici ax^2 + bx + cxy + dy + ey^2 = 1, kde [a,b,c,d,e] = coeffs. Pokud "plot=True", elipsu i vykreslime."""
  [a,b,c,d,e] = coeffs
  rng = np.random.default_rng(12345678912345678) ### tenhle radek zajistuje, ze ty nahodnz cisla jsou vygenerovanz pri kazdem behu stejne (lze smazat)
  theta_raw = rng.random(size=n_points)
  theta = np.sort(2*np.pi*theta_raw)
  cos_t, sin_t = np.cos(theta), np.sin(theta)
  A = a*(cos_t**2) + c*cos_t*sin_t + e*(sin_t**2)
  B = b*cos_t + d*sin_t
  r = (-B + np.sqrt(B**2 + 4*A)) / (2*A)
  xi = r*cos_t
  yi = r*sin_t
  if plot:
    plt.plot(np.append(xi,xi[0]),np.append(yi,yi[0]), marker='o', color='tab:blue', markeredgecolor='k', lw=1, label='Elipsa [{:.2f},{:.2f},{:.2f},{:.2f},{:.2f}]'.format(a,b,c,d,e))
    plt.gca().set_aspect("equal", "datalim")
  return xi,yi

### Vygenerujeme si "přesné" i "pozorované" body na nějaké konkretní elipse
n_points = 200
exact_coeffs = [1/2,0,0,0,1]
noise_scale = 1e-1*2
x_exact, y_exact = generate_exact_ellipse_points(exact_coeffs, n_points)
rng_x_noise = np.random.default_rng(987654321987654321) ### tenhle radek zajistuje, ze ty nahodnz cisla jsou vygenerovanz pri kazdem behu stejne (lze smazat)
rng_y_noise = np.random.default_rng(456132789456123798) ### tenhle radek zajistuje, ze ty nahodnz cisla jsou vygenerovanz pri kazdem behu stejne (lze smazat)
x_errs = rng_x_noise.standard_normal(size=n_points) * noise_scale
y_errs = rng_y_noise.standard_normal(size=n_points) * noise_scale
x_observed = x_exact + x_errs
y_observed = y_exact + y_errs

<div style="border-left: 4px solid #0969da; background-color: #ddf4ff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**📝 Úkol**

Sestavte matici $A$ a pravou stranu $\mathbf{b}$ z bodů `x_observed, y_observed` a vyřešte problém nejmenších čtverců; výsledek uložte do `fit_coeffs`.

</blockquote>
</div>

In [ ]:
### Sestavime a vyresime linearni problem nejmensich ctvercu
a1 = ...
a2 = ...
a3 = ...
a4 = ...
a5 = ...
A = ...
b = ...
fit_coeffs = ...

In [ ]:
### Vykreslovani
plt.figure(figsize=(8,5))
plt.plot(x_exact, y_exact, 's', color='tab:green', ms=12, alpha=.3, label='presne body')
plt.plot(x_observed, y_observed, 'o', color='tab:blue', ms=12, alpha=.3, label='zkreslene body')
_,_ = generate_exact_ellipse_points(fit_coeffs, n_points, plot=True)
plt.axis('equal'); plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.05),fancybox=True, shadow=True, ncol=3); plt.show()

### Úloha C2: [Okun's law](https://en.wikipedia.org/wiki/Okun%27s_law)

V makroekonomii existuje empirické pravidlo (aka vypozorovaná heuristika), tzv. Okun's law, které tvrdí, že růst HDP by měl být přímo úměrný poklesu nezaměstnanosti. Koeficienty této lineární závislosti se přirozeně můžou lišit v různých ekonomikách.

V praxi máme (odhadovaná) data o míře nezaměstnanosti a velikosti HDP, jeden kvartál za druhým. Pro ekonomiku USA v letech 1959-2015 jsou tato data dostupná, např. [zde, data.xls (sloupce 2 a 3)](https://www.rba.gov.au/publications/rdp/2015/2015-14-data.html). Nejprve si tedy data stáhneme.

In [ ]:
# URL naseho word-office souboru .xls
xls_url = "https://www.rba.gov.au/publications/rdp/2015/2015-14/additional-files/data.xls"
# Nazev pod kterym chceme soubor ulozit
local_filename = "data.xls"

print(f"Downloading {xls_url} to {local_filename} pomoci knihovny requests...")
response = requests.get(xls_url)
response.raise_for_status()
with open(local_filename, 'wb') as f:
    f.write(response.content)
print("Download hotov!")

Z popisu výše tedy máme stažena data odpovídající dvěma vektorům:
  - $\tilde{\mathbf{u}}$, který v sobě má data o míře nezaměstnanosti v USA v chronologickém pořadí (tj. $\tilde{u}_1$ je míra nezaměstnanosti v prvním čtvrtletí 1959, $\tilde{u}_2$ je míra nezaměstnanosti v druhém čtvrtletí 1959, atd.).
  - $\tilde{\mathbf{g}}$, který v sobě má data o velikosti HDP v USA v chronologickém pořadí (tj. $\tilde{g}_1$ je HDP v prvním čtvrtletí 1959, $\tilde{g}_2$ je HDP v druhém čtvrtletí 1959, atd.).

In [ ]:
df = pd.read_excel('data.xls', engine='xlrd')  # Predpokladam, ze slozka 'data.xls' je nahrana do tohoto googlecolabu, kdyztak si ji donahrajte (skrze ikonku slozky na liste uplne vpravo)
u_tilde = df.values[:,1].astype(float)
g_tilde = df.values[:,2].astype(float)

Hodnoty figurující v "Okun's law", tj. **procentuální změna HDP** a **změna míry nezaměstnanosti** si musíme sestrojit:
- $\mathbf{u} = [u_1, \dotsc, u_{m-1}], \quad \mathrm{kde} \; u_i = \tilde{u}_{i+1} - \tilde{u}_{i}$,
- $\mathbf{g} = [g_1, \dotsc, g_{m-1}], \quad \mathrm{kde} \; g_i = 100\cdot \tfrac{\tilde{g}_{i+1} - \tilde{g}_{i}}{\tilde{g}_{i}}$ (koeficient `100` změní 0.75 na 75%).

Nadto jsou data ve vektorech $\mathbf{u},\mathbf{g}$ seřazena chronologicky, zatímco vykreslování vztahu "$\mathbf{g}$ je funkce $\mathbf{u}$" s časem nesouvisí - chtěli bychom mít prvky $\mathbf{u}$ seřazeny podle velikosti.

Tohle všechno nám ošetří funkce `preprocess_data()` níže, která nám vrátí vektory $\mathbf{u}, \mathbf{g}$ odpovídající správně zprocesovaným datům.

<div style="border-left: 4px solid #0969da; background-color: #ddf4ff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**📝 Tužka a papír & kódění**

- **Tužka a papír**:
  - Matematicky formulujte lineární problém nejmenších čtverců pro nalezení koeficientů úměrnosti $\theta_0,\theta_1$, které udávají lineární závislost prvků $\mathbf{g}$ na $\mathbf{u}$, tj. které nejlépe odpovídají
  \begin{equation*} \mathbf{g}_i \approx \theta_0 + \theta_1 \cdot \mathbf{u}_i \quad i = 1,2,\dotsc , m,  \end{equation*}
  ve smyslu nejmenších čtverců, kde $m$ je počet dat (tj. párů $\mathbf{u}_i,\mathbf{g}_i$).
  - **Poznámka** : Tomuto problému se také říká [***lineární regrese***](https://en.wikipedia.org/wiki/Linear_regression).

- **Kódění**:
  - Vypočítejte parametry $\theta_0,\theta_1$ na základě odvozeného problému výše s pomocí jedné z našich metod,
    - ouput by měl být v poli `thetas`, tj. `thetas = np.array([theta_0,theta_1])`.

</blockquote>
</div>

In [ ]:
def preprocess_data(u_tilde,g_tilde):
  """ Odpovida data-preprocessing z namerenych dat na data, se kterymi chceme pracovat."""
  u = (u_tilde[1:]-u_tilde[:-1])
  g = 100*( (g_tilde[1:]-g_tilde[:-1])/g_tilde[1:] )
  sort_idx = np.argsort(u)
  u = u[sort_idx]
  g = g[sort_idx]
  return u,g

### data preprocessing
u,g = preprocess_data(u_tilde,g_tilde)

<div style="border-left: 4px solid #0969da; background-color: #ddf4ff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**📝 Úkol**

Sestavte matici $A$ z dat `u` a vyřešte problém nejmenších čtverců pro `thetas`.

</blockquote>
</div>

In [ ]:
### sestaveni a reseni linearnich nejmensich ctvercu
A = ...
b = g
thetas = ...

In [ ]:
### vykreslovani vysledku
plt.figure(figsize=(8,5))
plt.plot(u, g, 'o', ms=7, label='USA data, 1959-2015')
plt.plot(u, thetas[0] + thetas[1]*u, 'k-', label='Predikce pro data, ktera nezname')
plt.plot(u, thetas[0] + thetas[1]*u, marker='s', ms=7, lw=0, color="tab:gray", label='Aproximace pro data, ktera zname')
plt.xlabel(r'zmena nezamestnanosti (%)'); plt.ylabel(r'zmena HDP (%)'); plt.legend()
plt.show()

### ZÚ - zvládneme to lépe než pan Okun?

V úloze C2 jsme našli $\theta_0,\theta_1$, které nejlépe vysvětlují naše data $(\mathbf{u},\mathbf{g})$ ve smyslu

\begin{equation*} g_i \approx \theta_0 + \theta_1 u_i, \end{equation*}

tedy nejlepší lineární model pro pozorovaná data. Pro lineární model jsme se rozhodli "***zdánlivě náhodou*** - pan Okun nám to poradil". Jak by dopadly modely vyžívající polynomy vyššího stupně?

<div style="border-left: 4px solid #8250df; background-color: #fbefff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**❗ ZÚ (semestrální projekt)**

- **Tužka a papír**: Pro daný stupeň polynomu $d$ odvoďte matici $A$ a pravou stranu $\mathbf{b}$, které odpovídají modelu

  \begin{equation*} g_i \approx \theta_0 + \theta_1 u_i + \dotsc  + \theta_d u_i^d =: p_{\mathbf{\theta}}(u_i), \quad i=1,\dotsc ,m \end{equation*}

  ve smyslu nejmenších čtverců, kde $m$ je počet dat (tj. párů $\mathbf{u}_i,\mathbf{g}_i$) a $p_{\mathbf{\theta}}$ je hledaný polynom stupně nejvýše $d$ s koeficienty $\mathbf{\theta} = [\theta_0, \dotsc , \theta_d]$.
  - **Poznámka** : Tomuto problému se říká [***polynomiální regrese***](https://en.wikipedia.org/wiki/Polynomial_regression).

- **Kódění**:

  - Vypočítejte $\mathbf{\theta} = [\theta_0, \dotsc , \theta_d]$ ve smyslu nejmenších čtverců jednou z metod výše pro různá $d$
    - možná se bude hodit funkce [`np.vander()`](https://numpy.org/doc/2.2/reference/generated/numpy.vander.html)

  - na základě výsledků sestrojte polynomy $p_{\mathbf{\theta}}$ pomocí [`numpy.polynomial.polynomial.Polynomial()`](https://numpy.org/doc/2.2/reference/generated/numpy.polynomial.polynomial.Polynomial.html#numpy.polynomial.polynomial.Polynomial).

  - pomocí polynomů $p_{\mathbf{\theta}}$ můžeme (stejně jako v C2) buď
    - koukat na chybu aproximace v bodech, kde máme naměřená data (tj. právě v $\mathbf{u}_i$)
    - zkoušet predikovat změnu HDP pro "nové" hodnoty $v$, tj. $v \neq \mathbf{u}_i  \forall i$

  - Skript níže nám toto umožňuje pro tři různé hodnoty $d$ souběžně, ale není obtížné upravit vykreslování i pro více hodnot $d$ najednou.

  - Porovnejte různé volby modelů, tj. různá $d$. Porovnejte "kvalitu" aproximace a predikce. Doporučili byste na základě vašich výsledků na tomto data-setu "updated verzi Okun's law" nebo ne? Proč?

</blockquote>
</div>

In [ ]:
### Znovu pouzijeme "preprocess_data()" z bunky vyse.
u,g = preprocess_data(u_tilde,g_tilde)


### nejmensi ctverce pro polynomy vyssich stupnu
poly_degrees_to_try = np.array([1,5,9])
v_points = np.linspace(np.min(u), np.max(u), 1000)

# inicializace, abychom meli kam ukladat vysledky "g_approx := p_theta(u)" a "g_pred := p_theta(v_points)"
g_approx = np.zeros((len(poly_degrees_to_try),len(u)))
g_pred = np.zeros((len(poly_degrees_to_try),len(v_points)))

<div style="border-left: 4px solid #8250df; background-color: #fbefff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**❗ ZÚ (semestrální projekt)**

Pro každý stupeň polynomu `poly_degrees_to_try[ind]` sestavte Vandermondovu matici, vyřešte nejmenší čtverce pro `thetas` a sestrojte polynom `poly`; výsledky uložte do `g_approx`/`g_pred`.

</blockquote>
</div>

In [ ]:
# for cyklus pro jednotlive stupne polynomu
for ind in range(0,len(poly_degrees_to_try)):

  A = ...
  b = g
  thetas = ...
  poly = ...

  g_approx[ind,:] = ...
  g_pred[ind,:] = ...

In [ ]:
### vykreslovani vysledku
plt.figure(figsize=(20,10))

plt.subplot(2,3,1); plt.plot(u, g, 'o', label='USA data, 1959-2015'); plt.plot(u, g_approx[0,:], 's', lw=0, c="tab:green", label='d = {} : aproximace pro data, ktera zname'.format(poly_degrees_to_try[0]));
plt.legend(loc="upper center"); plt.ylabel(r'zmena HDP (%)');
plt.subplot(2,3,2); plt.plot(u, g, 'o', label='USA data, 1959-2015'); plt.plot(u, g_approx[1,:], 's', lw=0, c="orange", label='d = {} : aproximace pro data, ktera zname'.format(poly_degrees_to_try[1]));
plt.legend(loc="upper center");
plt.subplot(2,3,3); plt.plot(u, g, 'o', label='USA data, 1959-2015'); plt.plot(u, g_approx[2,:], 's', lw=0, c="tab:brown", label='d = {} : aproximace pro data, ktera zname'.format(poly_degrees_to_try[2]));
plt.legend(loc="upper center");

plt.subplot(2,3,4); plt.plot(u, g, 'o', label='USA data, 1959-2015'); plt.plot(v_points, g_pred[0,:], '-', c="tab:green", label='d = {} : predikce pro data, ktera nezname'.format(poly_degrees_to_try[0]));
plt.legend(loc="upper center"); plt.xlabel(r'zmena nezamestnanosti (%)'); plt.ylabel(r'zmena HDP (%)');
plt.subplot(2,3,5); plt.plot(u, g, 'o', label='USA data, 1959-2015'); plt.plot(v_points, g_pred[1,:], '-', c="orange", label='d = {} : predikce pro data, ktera nezname'.format(poly_degrees_to_try[1]));
plt.legend(loc="upper center"); plt.xlabel(r'zmena nezamestnanosti (%)');
plt.subplot(2,3,6); plt.plot(u, g, 'o', label='USA data, 1959-2015'); plt.plot(v_points, g_pred[2,:], '-', c="tab:brown", label='d = {} : predikce pro data, ktera nezname'.format(poly_degrees_to_try[2]));
plt.legend(loc="upper center"); plt.xlabel(r'zmena nezamestnanosti (%)');

plt.show()

## Hinty

<a name="head-w04-hinty-zua"></a>
### Hinty ZÚ

#### Vlastní čísla vs. singulární čísla matice $C$

Stačí pracovat s vlastními čísly matice $C$ nebo potřebujeme její singulární čísla? Vzpomeňte si na [Cviko 2](week-01-conditioning-stability).

#### Vlastní čísla blokové matice z definice

Vlastní čísla blokové matice lze počítat z definice, tj. $\lambda$ je vlastní číslo $C$ pokud $\det (C-\lambda I) = 0$.

#### Determinant blokové matice

Pokud jsou všechny výrazy definovány, pak platí
\begin{equation*}
\det \left( \begin{bmatrix} A & B \\ C & D \end{bmatrix} \right) =
\det \left(D\right) \det \left(A - CD^{-1}B\right) .
\end{equation*}

#### Taylorův rozvoj čísla podmíněnosti

Při odhadu asymptotického chování čísla podmíněnosti matice $C$ se může hodit Taylorův rozvoj (viz [2. cviko](week-01-conditioning-stability)).

<a name="head-w04-hinty-zu"></a>
### Hinty ZÚ

#### Odhad mocninné závislosti z log-log grafu

Předpokládejme, že máme funkci $y = g(x)$, kterou budeme vykreslovat. Pokud vidíme, že v `loglog()` vykreslení je graf $g$ rostoucí přímka s kladnými hodnotami, pak tedy platí
\begin{equation*} \log (g(x)) = \alpha \log(x)  + \beta , \end{equation*}
pro nějaké $\alpha, \beta > 0$. Jak pak vypadá $g(x)$?

#### Optimalizace maticového násobení v numpy

V `numpy` je kód pro maticové násobení detailně zoptimalizován - neplýtvá se ani kouskem dostupné paměti nebo možnostmi procesoru, protože na maticovém násobení je postavená ohromná spousta dalších algoritmů.

U funkce [`np.linalg.qr()`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.qr.html) už ta optimalizace není tak detailní.

<a name="head-w04-hinty-c1"></a>
### Hinty C1

#### Volba vektoru $\mathbf{b}$

Postupů je mnoho, ale podle mě ten nejpřirozenější povede na $\mathbf{b} = [1,\dotsc ,1]^T$.

#### Rozepsání [(4)](#eq4) jako skalární součin

Pro libovolný skalár $x_0$ platí následující šíkovné rozepsání $\alpha x_0^2 + \beta x_0 + \gamma = [x_0^2 \; x_0 \; 1] \begin{bmatrix} \alpha \\ \beta \\ \gamma \end{bmatrix} $. Zkuste si obdobně rozepsat [(4)](#eq4).